In [3]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

## 1、下载数据集

In [2]:
from datasets import load_dataset
ds = load_dataset("llamafactory/alpaca_zh", cache_dir="../data",split = "train")

In [3]:
ds

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 51155
})

In [4]:
ds = ds.train_test_split(test_size=0.8,seed = 42)
ds

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 10231
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 40924
    })
})

In [5]:
ds['train'][5]

{'instruction': '解释气候变化对环境的两个影响',
 'input': '',
 'output': '气候变化对环境有几个影响。其中一个影响是全球平均气温上升，导致冰川融化、极端天气事件和海平面上升。另一个影响是由极端天气条件和升温引起的自然栖息地破坏导致的生物多样性减少。'}

## 2、数据预处理

In [6]:
#模型下载
# from modelscope import snapshot_download
# model_dir = snapshot_download('qwen/Qwen2-0.5B', cache_dir="./models")

In [2]:
model_path = "./models/qwen/Qwen2-0___5B"

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer

Qwen2TokenizerFast(name_or_path='./models/qwen/Qwen2-0___5B', vocab_size=151643, model_max_length=32768, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [9]:
def process_func(datasets, tokenizer, max_length=256):
    """
    处理输入示例，合并用户的指令和输入字段，然后对输入和输出进行分词。
    该函数准备好训练自回归语言模型的数据格式。

    参数：
    - datasets (dict): 包含 'instruction'、'input' 和 'output' 字段的字典。
    - tokenizer (AutoTokenizer): 用于分词的 tokenizer。
    - max_length (int): 序列的最大长度。

    返回：
    - dict: 准备好的训练数据，包括 tokenized 的输入和标签。
    """
    # 1. 生成输入部分，包括 "Human: " 和 "Assistant: " 标签
    if datasets['input'].strip() == "":
        combined_input = "用户: " + datasets['instruction'] + "\n\n 机器人: "
    else:
        combined_input = "用户: " + datasets['instruction'] + "\n" + datasets['input'] + "\n\n 机器人: "

    # 2. 生成输出部分，并在最后加上终止标记
    output_text = datasets["output"] + tokenizer.eos_token  # 加上 eos_token 标记回复的结束

    # 3. 合并输入和输出作为模型的完整输入序列
    full_input = combined_input + output_text

    # 4. 对完整输入进行分词处理，确保生成 input_ids 和 attention_mask
    encodings = tokenizer(
        full_input, 
        max_length=max_length, 
        truncation=True, 
        padding="max_length", 
        return_tensors='pt'
    )

    # 5. 复制 input_ids 来生成标签
    labels = encodings["input_ids"].clone()

    # 6. 计算用户输入部分的长度（即需要忽略的部分）
    user_input_len = len(tokenizer(combined_input, truncation=True, max_length=max_length)["input_ids"])

    # 7. 将用户输入部分的标签设置为 -100，忽略这部分的损失
    labels[:, :user_input_len] = -100

    # 8. 返回 input_ids、attention_mask 和 labels，用于训练模型
    return {
        'input_ids': encodings['input_ids'].squeeze(0),  # 输入序列
        'attention_mask': encodings['attention_mask'].squeeze(0),  # 注意力掩码
        'labels': labels.squeeze(0)  # 标签，忽略用户输入部分
    }


In [10]:
# 假设 ds 是你的数据集，并且 tokenizer 已经定义
tokenized_ds = ds.map(
    process_func,
    remove_columns=['instruction', 'input', 'output'],  # 删除不需要的原始列
    fn_kwargs={'tokenizer': tokenizer},  # 将 tokenizer 传递给 process_func
)

tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10231
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 40924
    })
})

## 3、创建模型

In [4]:
model = AutoModelForCausalLM.from_pretrained(model_path)

### 3.1 设置微调参数

In [5]:
from peft import PromptTuningConfig, get_peft_model, TaskType, PromptTuningInit

# Hard Prompt
config = PromptTuningConfig(task_type=TaskType.CAUSAL_LM,
                            prompt_tuning_init=PromptTuningInit.TEXT,
                            prompt_tuning_init_text="根据用户和机器人的对话，学习生成文本",
                            num_virtual_tokens=20,
                            tokenizer_name_or_path=model_path)
config

PromptTuningConfig(peft_type=<PeftType.PROMPT_TUNING: 'PROMPT_TUNING'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, num_virtual_tokens=20, token_dim=None, num_transformer_submodules=None, num_attention_heads=None, num_layers=None, prompt_tuning_init=<PromptTuningInit.TEXT: 'TEXT'>, prompt_tuning_init_text='根据用户和机器人的对话，学习生成文本', tokenizer_name_or_path='./models/qwen/Qwen2-0___5B', tokenizer_kwargs=None)

### 3.2 重新载入模型

In [6]:
model = get_peft_model(model, config)
model

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


PeftModelForCausalLM(
  (base_model): Qwen2ForCausalLM(
    (model): Qwen2Model(
      (embed_tokens): Embedding(151936, 896)
      (layers): ModuleList(
        (0-23): 24 x Qwen2DecoderLayer(
          (self_attn): Qwen2SdpaAttention(
            (q_proj): Linear(in_features=896, out_features=896, bias=True)
            (k_proj): Linear(in_features=896, out_features=128, bias=True)
            (v_proj): Linear(in_features=896, out_features=128, bias=True)
            (o_proj): Linear(in_features=896, out_features=896, bias=False)
            (rotary_emb): Qwen2RotaryEmbedding()
          )
          (mlp): Qwen2MLP(
            (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
            (up_proj): Linear(in_features=896, out_features=4864, bias=False)
            (down_proj): Linear(in_features=4864, out_features=896, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): Qwen2RMSNorm()
          (post_attention_layernorm): Qwen2RMSNorm(

In [7]:
model.prompt_encoder

ModuleDict(
  (default): PromptEmbedding(
    (embedding): Embedding(20, 896)
  )
)

### 3.3 查看参数量

In [14]:
model.print_trainable_parameters()

trainable params: 17,920 || all params: 494,050,688 || trainable%: 0.0036


## 4、创建训练参数

In [15]:
args = TrainingArguments(
    output_dir="./models_for_chatbot_hard,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,  # 每4步累计一次
    per_device_eval_batch_size=32,
    logging_steps=20,
    num_train_epochs=1,
    report_to=['tensorboard'],
)

## 5、创建训练器并训练

In [16]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"], 
    eval_dataset=tokenized_ds["test"],     
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
)

In [17]:
trainer.train()

  0%|          | 0/319 [00:00<?, ?it/s]

{'loss': 14.8154, 'grad_norm': 13.340837478637695, 'learning_rate': 4.686520376175549e-05, 'epoch': 0.06}
{'loss': 14.0202, 'grad_norm': 15.68494987487793, 'learning_rate': 4.373040752351097e-05, 'epoch': 0.13}
{'loss': 12.9329, 'grad_norm': 17.269947052001953, 'learning_rate': 4.059561128526646e-05, 'epoch': 0.19}
{'loss': 11.9679, 'grad_norm': 21.80983543395996, 'learning_rate': 3.746081504702195e-05, 'epoch': 0.25}
{'loss': 10.5892, 'grad_norm': 48.35317611694336, 'learning_rate': 3.4326018808777435e-05, 'epoch': 0.31}
{'loss': 8.7714, 'grad_norm': 35.92039489746094, 'learning_rate': 3.119122257053292e-05, 'epoch': 0.38}
{'loss': 6.8759, 'grad_norm': 69.11973571777344, 'learning_rate': 2.8056426332288405e-05, 'epoch': 0.44}
{'loss': 5.0077, 'grad_norm': 66.71023559570312, 'learning_rate': 2.4921630094043887e-05, 'epoch': 0.5}
{'loss': 3.1597, 'grad_norm': 68.65534210205078, 'learning_rate': 2.1786833855799376e-05, 'epoch': 0.56}
{'loss': 1.8762, 'grad_norm': 55.476505279541016, 'lea

TrainOutput(global_step=319, training_loss=5.99368075367799, metrics={'train_runtime': 814.68, 'train_samples_per_second': 12.558, 'train_steps_per_second': 0.392, 'total_flos': 5611659152326656.0, 'train_loss': 5.99368075367799, 'epoch': 0.9976544175136826})